# Make a song

This makes a full song from your words. It runs on Kaggle's computer, not yours, and it is free.

## Set two things first

Look at the panel on the right side of the screen.

1. Find **Accelerator** and set it to **GPU T4 x2**
2. Find **Internet** and turn it **On**

If you skip these it will not work.

## Then

Click **Run All** at the top. The first time takes about 10 minutes because it has to fetch
the music program. After that, songs take a few minutes each.

When it is done there is a play button near the bottom, and the song file shows up in the
**Output** panel on the right so you can download it.

## To write your own song

Scroll to the box that says **YOUR SONG**. Change the words in there. That is the only box
you ever need to touch. Then click Run All again.

Kaggle gives you 30 hours of this a week, free. A song is a few minutes, so that is a lot
of songs.


## Step 1: check the computer

Makes sure Kaggle gave you a graphics card. If it complains, set **Accelerator** to **GPU T4 x2** in the right panel and start over.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| GPUs", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2' and restart the session."


## Step 2: fetch the music program

Takes a couple of minutes. Nothing to do here, just let it run.

In [ ]:
import os, subprocess, sys

COMFY = "/kaggle/temp/ComfyUI"
if not os.path.exists(COMFY):
    !git clone --depth 1 --branch v0.33.1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}

!pip install -q -r {COMFY}/requirements.txt 2>&1 | tail -5
print("ComfyUI ready at", COMFY)


## Step 3: fetch the music brain

About 12 GB. This is the part that actually knows how to write and sing music. Give it a few
minutes.

It only comes down once. Lyricist saves it into your Kaggle account the first time
(a notebook called **Lyricist Model Cache**) and plugs it straight in after that.

You do **not** need a Hugging Face account for this. If you happen to have saved a token in
Kaggle's Secrets under the name `huggingface`, it will use it. If not, it carries on fine.

In [ ]:
from huggingface_hub import hf_hub_download
import os, shutil

# A Hugging Face token is OPTIONAL and changes nothing about the song. It only
# stops the Hub throttling anonymous downloads, and this fetches 12 GB.
#
# Lyricist puts the token from its Settings on the line below when you have one
# saved, so this notebook holds a copy of it. That is fine while the notebook is
# private and it is yours. Two things follow from it: do not make this notebook
# public, and if you ever want the copy gone, clear the token in Lyricist's
# Settings and press Make the song once, which overwrites this line with an
# empty one.
#
# If you would rather it was never written here at all, leave Settings empty and
# use Kaggle Secrets instead: Add-ons, then Secrets, label it exactly
# "huggingface", and switch it on for this notebook.
HF_TOKEN = ""   # filled in by Lyricist when you have one saved

try:
    from huggingface_hub import login
    if HF_TOKEN.strip():
        login(HF_TOKEN.strip())
        print("Signed in to Hugging Face with the token from Lyricist.")
    else:
        from kaggle_secrets import UserSecretsClient
        login(UserSecretsClient().get_secret("huggingface"))
        print("Signed in to Hugging Face with your Kaggle secret.")
except Exception:
    print("No Hugging Face token. Not needed, the download is just slower.")

COMFY = "/kaggle/temp/ComfyUI"
REPO = "Comfy-Org/MiniMax-Music-3"
FILES = [
    ("diffusion_models/minimax_music3_dit_int8_convrot.safetensors",           "diffusion_models"),
    ("text_encoders/minimax_music3_text_encoder_pruned_int8_convrot.safetensors", "text_encoders"),
    ("vae/minimax_music3_dav.safetensors",                                     "vae"),
]

# THE 12 GB COMES DOWN ONCE, NOT EVERY SONG. Lyricist saves the model into your
# own Kaggle account the first time (a notebook called "Lyricist Model Cache")
# and attaches it here from then on. Kaggle mounts it under /kaggle/input, so it
# is linked straight in. Anything not there is downloaded like before.
def saved_copy(name):
    for root, _, names in os.walk("/kaggle/input"):
        if name in names:
            return os.path.join(root, name)
    return None

for remote, folder in FILES:
    dest_dir = f"{COMFY}/models/{folder}"
    os.makedirs(dest_dir, exist_ok=True)
    name = os.path.basename(remote)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest):
        print("already have", name); continue
    if os.path.lexists(dest):
        os.remove(dest)   # a link to a copy that is gone
    p = saved_copy(name)
    if p:
        print("using the saved copy of", name)
    else:
        print("downloading", remote, "...")
        p = hf_hub_download(repo_id=REPO, filename=remote, cache_dir="/kaggle/temp/hf")
    os.symlink(p, dest)
    print("  ->", dest, f"{os.path.getsize(p)/1e9:.2f} GB")

print("\nAll weights in place.")


---
# YOUR SONG
---

**This is the only box you ever change.**

**CAPTION** is how you describe the sound. Say the style, the speed, the instruments, and what
the singer sounds like. The more you tell it, the closer you get. Keep the three headings.

**LYRICS** are your words. Mark the parts with square brackets so it knows the shape of the
song: `[Intro]` `[Verse]` `[Chorus]` `[Bridge]` `[Instrumental]` `[Outro]`.
For an instrumental, delete the lyrics and say "instrumental, no vocals" in the caption.

**DURATION** is how long the song is, in seconds. 300 is the longest (five minutes).
Try 60 first so you can hear how it sounds without a long wait.

**SEEDS** is which versions you get. Each number is one take. The same words with a different
number gives you a completely different performance of the same song, so `[222, 777]` makes two
different takes and hands you both. Kaggle's free machine has two graphics cards, so two takes
finish in about the time one used to. Four is the most it will do at once.

**STEPS** is how much care it puts in. 30 is normal. 50 sounds a bit better and takes longer.

**CFG** is how closely it sticks to your description. 1.7 is normal. Higher follows your words
harder, lower lets it wander.

In [ ]:
CAPTION = """Global Metadata: Gritty blues rock, 96 BPM, E minor. Raw and driving, a dusty roadhouse stomp that builds through the chorus.

Vocal Details: Gravelly male lead, weathered and full-throated, sung slightly ahead of the beat with real bite. Rough gang-vocal answers on the chorus.

Arrangement: Dirty overdriven electric guitar, thumping kick and snare, walking bass, tambourine, a wailing slide lead over the chorus."""

LYRICS = """[Intro]

[Verse]
Steel in my hands and the sun going down
Sparks on the deck of a nameless town
I built every beam that you drive on now
Nobody asks me how

[Chorus]
So I sing it loud, I sing it free
Every road out here belongs to me
Ain't got a dollar but I got my say
And I'm singing it anyway

[Verse]
Hands full of callus and a voice full of smoke
Every song I ever wrote came from something broke
They can keep the ceiling, I'll keep the sky
I ain't done yet, and here's why

[Chorus]
So I sing it loud, I sing it free
Every road out here belongs to me
Ain't got a dollar but I got my say
And I'm singing it anyway

[Outro]
"""

DURATION = 60           # seconds, up to 300
SEEDS    = [222, 777]   # one number, one song. Two numbers, two different takes at the same time.
STEPS    = 30           # diffusion steps. 30 is the shipped recipe; try 50 for more polish
CFG      = 1.7


## Step 4: start it up

Nothing to do. It is warming up in the background.

In [ ]:
import subprocess, time, urllib.request, os, sys

COMFY = "/kaggle/temp/ComfyUI"
LOG = "/kaggle/temp/comfy.log"

def comfy_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)
        return True
    except Exception:
        return False

if not comfy_up():
    logf = open(LOG, "wb")
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
                      "--disable-auto-launch"],
                     cwd=COMFY, stdout=logf, stderr=subprocess.STDOUT)
    for i in range(180):
        if comfy_up():
            print(f"ComfyUI up after {i}s"); break
        time.sleep(1)
    else:
        print("ComfyUI did not start. Last 40 log lines:")
        print(open(LOG, errors="ignore").read()[-4000:])
        raise SystemExit(1)
else:
    print("ComfyUI already running")


## Step 5: write the song

This is the part that takes a few minutes. You will see a timer counting up.

The first long stretch is it working out the tune and the singing. It looks stuck. It is not. Let it go.

In [ ]:
import json, urllib.request, time, uuid, os, shutil, subprocess, sys, threading

S_BASE  = "http://127.0.0.1"
COMFY   = "/kaggle/temp/ComfyUI"
WORKING = "/kaggle/working"

try:
    seeds = [int(s) for s in SEEDS]
except Exception:
    seeds = [int(SEEDS)]
seeds = seeds[:4] or [222]

# TWO TAKES AT THE SAME TIME, NOT ONE AFTER THE OTHER.
# Kaggle's free machine is a T4 x2: two whole graphics cards. One ComfyUI only
# ever uses the first of them, so the second card sat idle through every song.
# Now a second ComfyUI is started on the second card and the takes are split
# between the two, which makes two versions cost about what one used to.
# It degrades honestly: one card, or a second server that will not come up, and
# every take runs through the first server one after another. Nothing is lost
# but the speed, and you still get every take you asked for.
def _up(port):
    try:
        urllib.request.urlopen(f"{S_BASE}:{port}/system_stats", timeout=3)
        return True
    except Exception:
        return False

def _start(port, gpu, wait=240):
    if _up(port):
        return True
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/temp/comfy-{port}.log", "wb")
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(port),
                      "--disable-auto-launch"],
                     cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(wait):
        if _up(port):
            return True
        time.sleep(1)
    return False

def _graph(seed, tag):
    return {
        "1": {"class_type": "UNETLoader",
              "inputs": {"unet_name": "minimax_music3_dit_int8_convrot.safetensors",
                         "weight_dtype": "default"}},
        "2": {"class_type": "CLIPLoader",
              "inputs": {"clip_name": "minimax_music3_text_encoder_pruned_int8_convrot.safetensors",
                         "type": "minimax", "device": "default"}},
        "3": {"class_type": "VAELoader",
              "inputs": {"vae_name": "minimax_music3_dav.safetensors"}},
        "4": {"class_type": "MiniMaxMusic3TextEncode",
              "inputs": {"clip": ["2", 0], "caption": CAPTION, "lyrics": LYRICS,
                         "seed": seed, "max_duration": float(DURATION),
                         "cfg_scale": CFG, "top_k": 50}},
        "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
        "6": {"class_type": "EmptyMiniMaxMusic3LatentAudio",
              "inputs": {"seconds": ["4", 1], "batch_size": 1}},
        "7": {"class_type": "KSampler",
              "inputs": {"model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0],
                         "latent_image": ["6", 0], "seed": seed, "steps": STEPS, "cfg": CFG,
                         "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}},
        # 16 GB holds this comfortably, so plain decode: no tiling, no seam risk
        "8": {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
        "9": {"class_type": "SaveAudioAdvanced",
              "inputs": {"audio": ["8", 0], "filename_prefix": tag, "format": "flac"}},
    }

def _submit(port, graph):
    req = json.dumps({"prompt": graph, "client_id": str(uuid.uuid4())}).encode()
    return json.load(urllib.request.urlopen(f"{S_BASE}:{port}/prompt", req))["prompt_id"]

def _collect(port, pid, tag, limit=21600):
    """Wait for one prompt and copy whatever audio it made into /kaggle/working."""
    t0 = time.time()
    while time.time() - t0 < limit:
        try:
            h = json.load(urllib.request.urlopen(f"{S_BASE}:{port}/history/{pid}", timeout=30))
        except Exception:
            time.sleep(10)
            continue
        if pid not in h:
            time.sleep(10)
            continue
        for m in h[pid].get("status", {}).get("messages", []):
            if m[0] == "execution_error":
                print(f"\n{tag} ERROR:", json.dumps(m[1], indent=1)[:1200])
        made = []
        for o in h[pid].get("outputs", {}).values():
            for a in o.get("audio", []):
                src = os.path.join(COMFY, "output", a.get("subfolder", ""), a["filename"])
                dst = os.path.join(WORKING, a["filename"])
                if os.path.exists(src):
                    shutil.copy(src, dst)
                    made.append(dst)
                    print(f"  SAVED -> {dst}")
        return made
    print(f"  {tag} timed out.")
    return []

try:
    import torch
    cards = torch.cuda.device_count()
except Exception:
    cards = 1

ports = [8188]
if len(seeds) > 1 and cards > 1:
    print("Two graphics cards here. Starting a second one so the takes run side by side.")
    if _start(8189, 1):
        ports.append(8189)
    else:
        print("The second card did not come up, so the takes will run one after the other.")
elif len(seeds) > 1:
    print(f"{cards} graphics card here, so the takes run one after the other.")

if not _up(8188):
    _start(8188, 0)

t0 = time.time()
lanes = {p: [] for p in ports}
for i, seed in enumerate(seeds):
    lanes[ports[i % len(ports)]].append((i + 1, seed))

out_files = []
lock = threading.Lock()

def _run_lane(port, jobs):
    for n, seed in jobs:
        # The run number goes in the name: the same seed on a new song must not
        # share a file name with the last one, or the old song is kept instead.
        tag = f"take{n}_seed{seed}" + (f"_run{RUN}" if globals().get("RUN") else "")
        print(f"Take {n} (seed {seed}) started on port {port}.")
        try:
            pid = _submit(port, _graph(seed, tag))
        except Exception as e:
            print(f"  take {n} could not be sent: {e}")
            continue
        made = _collect(port, pid, tag)
        with lock:
            out_files.extend(made)

print(f"Making {len(seeds)} take(s) of {DURATION}s. This is the slow part, let it run.\n")
threads = [threading.Thread(target=_run_lane, args=(p, lanes[p]), daemon=True)
           for p in ports if lanes[p]]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    print(f"  {time.time()-t0:6.0f}s elapsed", end="\r")
    time.sleep(10)
for t in threads:
    t.join()

el = time.time() - t0
print(f"\nDone in {el:.0f}s ({el/60:.1f} min). {len(out_files)} file(s) in /kaggle/working.")
for f in sorted(out_files):
    print(" ", f)


## Step 6: listen

Hit play below.

To keep it: look at the **Output** panel on the right side of the screen and download the file from there.

In [ ]:
from IPython.display import Audio, display

if out_files:
    for f in out_files:
        print(f, f"{os.path.getsize(f)/1e6:.1f} MB")
        display(Audio(f))
else:
    print("No audio produced - check the error above, and the log:")
    print(open("/kaggle/temp/comfy.log", errors="ignore").read()[-3000:])


---
## A few things worth knowing

**Making another song is fast.** Change the words in YOUR SONG, then run just the last two
boxes. It does not have to fetch anything again, so you skip straight to the music.

**The description matters more than any setting.** "A sad song" gives you mush. "Slow country
ballad, 70 BPM, brushed drums, pedal steel, weary older male voice close to the mic" gives you
something you can use. Name the instruments. Name the mood. Say what the voice sounds like.

**If a song comes out wrong**, change SEED first before you change anything else. Often the
words were fine and you just got a bad take.

**If the session disconnects**, Kaggle stops after 9 hours or if you leave it idle too long.
Start it again and run from the top.

**One legal note:** if you ever sell anything made with this, the makers require the name
"MiniMax-Music3" shown somewhere in your product. That only kicks in for commercial use.
